# Lesson 3: Building good datasets

Companion to `curriculum/03-datasets.md`.

**Big idea:** your eval can only be as good as its cases. Most of the value, and most of the work, is in
choosing *what to test*.

Mock model only: free, offline, no API key. Run cells top to bottom.
**🧠 Check** = answer in your own words before moving on. **✍️ Your turn** = cells for you to write (ask Claude for a hint).

> Your thread through the course: an AI **validator** that reads an Issue from the GRC platform and checks
> its rating against the Incident & Issue Management Standard.

In [ ]:
import json, random, re
from collections import Counter, defaultdict

from evalkit import app, graders
from evalkit.stats import bootstrap_ci, mean, paired_bootstrap_delta, wilson, cohens_kappa
from nb_helpers import DEV, HOLDOUT, load, run, report, failures, mock_task

dev = load(DEV)
holdout = load(HOLDOUT)
print(f"dev={len(dev)} cases, holdout={len(holdout)} cases")

## 1. Where cases come from (best to worst)

1. **Real production data** (anonymised). For you: historical Issues from the GRC platform.
2. **Real failures**: Issues that 2nd line or audit *re-rated*. These are your highest-value cases.
3. **Expert-written cases** that target known hard spots.
4. **Synthetic cases**: useful for coverage, but cleaner and more uniform than reality. Always review them by hand.

## 2. Coverage: think in dimensions

List the ways inputs vary, then check you have cases across each one. What does the dev set cover?

In [ ]:
tag_counts = Counter(t for c in dev for t in c["tags"])
prio_counts = Counter(c["expected"]["priority"] for c in dev)
cat_counts = Counter(c["expected"]["category"] for c in dev)

print("expected priority:", dict(prio_counts))
print("expected category:", dict(cat_counts))
print("\ntags with only ONE case (can't conclude anything from these):")
print("  ", sorted(t for t, n in tag_counts.items() if n == 1))

### 🧠 Check
With a single `security` case, the mock scores either 0% or 100% on that slice. What would you need before
saying *"we handle security tickets well"*?

## 3. Shortcuts: could a dumb baseline score well?

If a trivial rule scores high, your dataset can't tell a smart system from a dumb one. Here's the
**majority-class baseline** for priority: always predict the most common label.

In [ ]:
most_common = prio_counts.most_common(1)[0][0]
hits = sum(most_common == c["expected"]["priority"]
           or most_common in c["expected"].get("acceptable", {}).get("priority", [])
           for c in dev)
print(f"always predicting '{most_common}': {hits}/{len(dev)} = {hits/len(dev):.0%} priority accuracy")

s = report(run(dev), label="mock model")
print("\n(Yes: the dumb baseline beats the mock on priority.)")

The mock model is just **keyword rules**, and it scores about 95% on *category*. That tells you the dev
set's category labels are too keyword-aligned to test real understanding.

### ✍️ Your turn
Compute the majority-class baseline for **category**. Then think about your validator: if most historical
Issues are rated *Medium* and the assigned rating is usually right, what does an "always agree with the
assigned rating" baseline score? Why is that a trap?

In [ ]:
# ✍️ Your code here: majority-class baseline for category

## 4. Ground truth is harder than it looks

**Disagreement among labellers is a signal about your spec, not noise to average away.**
**Cohen's kappa** measures agreement beyond chance (1 = perfect, 0 = no better than chance, > 0.6 decent).

Simulated example: two risk reviewers independently rate the same 20 Issues.

In [ ]:
reviewer_a = ["Low","Medium","Medium","High","High","Medium","Low","Very High","High","Medium",
              "Medium","High","Low","Medium","High","Very High","Medium","Low","High","Medium"]
reviewer_b = ["Low","Medium","High","High","Very High","Medium","Low","Very High","High","High",
              "Medium","High","Medium","Medium","Very High","Very High","Medium","Low","High","High"]

agree = mean(a == b for a, b in zip(reviewer_a, reviewer_b))
print(f"raw agreement: {agree:.0%}   kappa: {cohens_kappa(reviewer_a, reviewer_b):.2f}\n")
print("disagreements:")
for i, (a, b) in enumerate(zip(reviewer_a, reviewer_b)):
    if a != b:
        print(f"  issue {i:>2}: A={a:<9} B={b}")

### 🧠 Check
Every disagreement above sits between **adjacent** levels, and B rates higher than A every time. What does
that suggest about the Standard's wording, or about the reviewers? And what's the **ceiling** for your
validator's agreement with *either* reviewer?

**Two more traps:**
- **Label errors.** Re-derive a sample of labels yourself.
- **Labels produced by the model.** If "expected" came from the current AI, the eval rewards *imitation*,
  not correctness. For you: are historical ratings "gold", or are they what you're trying to correct?

## 5. Dev vs holdout

- **Dev set**: you look at it and iterate on it.
- **Holdout set**: you run it rarely, and never tune against it. It catches **overfitting** (Lesson 6).

In [ ]:
_ = report(run(dev), label="dev")
_ = report(run(holdout), label="holdout")

## 6. How many cases?

Rules of thumb (convention, not law): **20–50** to start and catch big regressions; **100–300** to detect
5–10 point changes. Coverage of important slices beats raw count.

### ✍️ Your turn: coverage plan for Issues
List 5–7 **dimensions** your Issue dataset must cover (e.g. rating level, business unit, impact type,
borderline vs clear-cut, Issues later re-rated, free-text quality…), and a target count per slice.

*(your coverage plan here)*

---
## As a lead engineer…

- **Ownership:** who owns the golden set of confirmed ratings? It needs an owner and a review process, like a schema.
- **Labelling guidelines:** written rules for "this is High because…". For you, the Standard *is* the start of this, but you'll find gaps in it.
- **Privacy:** Issue data in an eval set needs the same care as in production.
- **Failure → test case:** every Issue the validator gets wrong in production becomes a new case.

### 🧠 Discussion
Your team has 4 engineers and no labellers. Where do the first 50 confirmed ratings come from, and whose time
do you need to ask for?

**Recap**
- Real data and real failures make the best cases. Cover dimensions, not just volume.
- Check baselines: if a dumb rule scores well, the dataset is too easy.
- Human disagreement is spec feedback, and it caps what any validator can achieve.

**Next:** `/lab 2`, then `/lesson 4`.